# <span style="color:green">**Agentes en LangChain v1.0**</span>

##### <span style="color:red">**Nota:** Este es un módulo "ipynb" creado por el **[Ing. Kevin Inofuente Colque](https://www.linkedin.com/in/kevin-inofuente-colque/)** de <span style="color:orange">**DataPath**</span> . Con mucho aprecio para mis colegas AI Enginners en toda Latinoamérica.</span> 

##### <span style="color:red">**Nota 2:** Algunos términos pueden estar en portuñol</span>

## ¿Qué es un Agente?

Un **agente** es un sistema que usa un LLM como "motor de decisión" para elegir qué hacer paso a paso. A diferencia de una llamada normal a un modelo (que solo responde texto), un agente puede:

- 🛠️ **Usar herramientas (tools)**: funciones que tú defines (consultar una API, hacer cálculos, buscar en internet, etc.).
- 🔁 **Razonar en bucle**: el modelo decide qué tool llamar, ve el resultado, y decide el siguiente paso.
- 🧠 **Mantener contexto** entre pasos de la misma conversación.

En LangChain v1.0 hay dos funciones principales para crear agentes:

| Función | Cuándo usarla |
|---|---|
| `create_agent` | Agentes "normales" — un LLM + herramientas + bucle. **Cubre el 90% de los casos.** |
| `create_deep_agent` | Agentes "profundos" estilo Claude Code / Deep Research, con planning, file system y sub-agentes. |

## <span style="color:orange">**Setup: Variables de Entorno**</span>

In [1]:
from dotenv import load_dotenv, find_dotenv
import os

# Cargar variables del archivo .env
_ = load_dotenv(find_dotenv())

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("La clave OPENAI_API_KEY no está configurada en el archivo .env.")
print("API Key de OpenAI cargada correctamente.")

API Key de OpenAI cargada correctamente.


## <span style="color:orange">**`create_agent` — Agentes en LangChain v1.0**</span>

`create_agent` es la **función estándar en v1.0** para crear un agente. Reemplaza al antiguo `AgentExecutor`. Recibe un modelo y una lista de herramientas, y devuelve un agente listo para invocar.

Internamente está construido sobre **LangGraph**, por lo que el agente ya viene con:

- ✅ Streaming de respuestas
- ✅ Persistencia y checkpointing (memoria entre invocaciones)
- ✅ Manejo automático del bucle "razonar → llamar tool → ver resultado → razonar de nuevo"

**Los 3 pasos:**

1. Definir herramientas con el decorador `@tool`.
2. Crear el agente con `create_agent(model, tools, system_prompt)`.
3. Invocarlo con un mensaje del usuario.

### <span style="color:skyblue">**Paso 1: Definir las herramientas (`@tool`)**</span>

Una herramienta es una función de Python normal **decorada con `@tool`**. La docstring y los type hints son **muy importantes**: el modelo los lee para saber qué hace la tool y cuándo usarla.

In [2]:
from langchain.tools import tool

@tool
def obtener_clima(ciudad: str) -> str:
    """Devuelve el clima actual de una ciudad dada."""
    # En un caso real aquí harías una llamada HTTP a una API del clima.
    # Para este ejemplo simulamos la respuesta.
    climas = {
        "Madrid": "Soleado, 24°C",
        "Lima": "Nublado, 18°C",
        "Buenos Aires": "Lluvioso, 15°C",
    }
    return climas.get(ciudad, f"No tengo datos del clima para {ciudad}.")

@tool
def sumar(a: float, b: float) -> float:
    """Suma dos números y devuelve el resultado."""
    return a + b

# Las tools son objetos especiales — podemos ver su nombre y descripción
print(f"Tool: {obtener_clima.name}")
print(f"Descripción: {obtener_clima.description}")

Tool: obtener_clima
Descripción: Devuelve el clima actual de una ciudad dada.


/opt/anaconda3/envs/LangChain-Introduccion-2026/lib/python3.11/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### <span style="color:skyblue">**Paso 2: Crear el agente con `create_agent`**</span>

In [3]:
from langchain.agents import create_agent

agente = create_agent(
    model="openai:gpt-4.1",                  # init_chat_model se llama por dentro
    tools=[obtener_clima, sumar],            # lista de herramientas que el agente puede usar
    system_prompt="Eres un asistente útil que responde preguntas usando las herramientas disponibles.",
)

print("✅ Agente creado")

✅ Agente creado


### <span style="color:skyblue">**Paso 3: Invocar el agente**</span>

El agente se invoca pasando un diccionario con la clave `messages`. Devuelve otro diccionario que contiene **toda la conversación** (incluyendo las llamadas internas a tools).

In [4]:
resultado = agente.invoke({
    "messages": [
        {"role": "user", "content": "¿Qué clima hace en Madrid? Y de paso, ¿cuánto es 25 + 17?"}
    ]
})

# La respuesta final del agente está en el último mensaje
respuesta_final = resultado["messages"][-1].content
print("Respuesta final del agente:")
print(respuesta_final)

Respuesta final del agente:
En Madrid hace un clima soleado con una temperatura de 24°C. Además, la suma de 25 + 17 es igual a 42.


### <span style="color:skyblue">**Ver el flujo completo de mensajes**</span>

Si recorremos todos los mensajes del resultado, podemos ver **cómo razonó el agente**: qué tools llamó, con qué argumentos, y qué le devolvieron.

In [5]:
for i, mensaje in enumerate(resultado["messages"]):
    print(f"--- Mensaje {i+1}: {type(mensaje).__name__} ---")
    if mensaje.content:
        print(f"Contenido: {mensaje.content}")
    if hasattr(mensaje, 'tool_calls') and mensaje.tool_calls:
        for tc in mensaje.tool_calls:
            print(f"🛠️  Tool llamada: {tc['name']}({tc['args']})")
    print()

--- Mensaje 1: HumanMessage ---
Contenido: ¿Qué clima hace en Madrid? Y de paso, ¿cuánto es 25 + 17?

--- Mensaje 2: AIMessage ---
🛠️  Tool llamada: obtener_clima({'ciudad': 'Madrid'})
🛠️  Tool llamada: sumar({'a': 25, 'b': 17})

--- Mensaje 3: ToolMessage ---
Contenido: Soleado, 24°C

--- Mensaje 4: ToolMessage ---
Contenido: 42.0

--- Mensaje 5: AIMessage ---
Contenido: En Madrid hace un clima soleado con una temperatura de 24°C. Además, la suma de 25 + 17 es igual a 42.



## <span style="color:orange">**`create_deep_agent` — Agentes Profundos**</span>

`create_deep_agent` viene del paquete **`deepagents`** (LangChain Labs). Es una versión **"todo incluido"** de un agente, pensada para tareas largas y complejas tipo **Claude Code** o **Deep Research**.

**¿Qué agrega `create_deep_agent` sobre `create_agent`?**

- 📋 **Planificación automática**: el agente puede escribir y actualizar una lista de tareas (`write_todos`).
- 📁 **Sistema de archivos virtual**: puede leer, escribir y editar archivos durante la conversación.
- 🤖 **Sub-agentes**: puede delegar subtareas a otros agentes con contexto aislado.
- 🧠 **Manejo automático de contexto largo**: resume cuando la conversación crece.
- 📝 **Prompts del sistema afinados** para razonamiento profundo.

### <span style="color:skyblue">**Instalación**</span>

Si aún no tienes el paquete `deepagents`, instálalo:

```bash
pip install deepagents
```

### <span style="color:skyblue">**Ejemplo simple con `create_deep_agent`**</span>

Reutilizamos las mismas tools de antes (`obtener_clima` y `sumar`). El agente "profundo" las usará pero **además** tiene acceso a sus tools internas (planning, archivos, sub-agentes).

In [6]:
from deepagents import create_deep_agent

agente_profundo = create_deep_agent(
    model="openai:gpt-4.1",
    tools=[obtener_clima, sumar],
    system_prompt=(
        "Eres un asistente de investigación meteorológica. "
        "Cuando recibas una tarea compleja, primero crea un plan con write_todos, "
        "luego ejecuta cada paso usando las herramientas disponibles."
    ),
)

print("✅ Agente profundo creado")

✅ Agente profundo creado


In [7]:
resultado_profundo = agente_profundo.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Investiga el clima en Madrid, Lima y Buenos Aires. "
                "Para cada ciudad, dime el clima y luego dame un resumen final."
            ),
        }
    ]
})

# Respuesta final
print("Respuesta final del agente profundo:")
print(resultado_profundo["messages"][-1].content)

Respuesta final del agente profundo:
[{'type': 'text', 'text': 'Clima actual en cada ciudad:\n- Madrid: Soleado, 24°C\n- Lima: Nublado, 18°C\n- Buenos Aires: Lluvioso, 15°C\n\nResumen: Hoy en Madrid predomina el sol y temperaturas cálidas, Lima presenta nubes y clima templado, mientras que Buenos Aires está con lluvias y temperatura más fresca. Cada ciudad muestra un clima diferente: sol, nubes o lluvia.', 'annotations': [], 'id': 'msg_0bfa97f4579a8113006a028b661ea08195ae1973fe382a837d'}]


## <span style="color:orange">**¿Cuándo usar cada uno?**</span>

| Caso de uso | Recomendado |
|---|---|
| Chatbot que llama una o dos APIs | `create_agent` |
| Asistente que responde con tools simples | `create_agent` |
| Q&A sobre una base de conocimiento (RAG con tools) | `create_agent` |
| Asistente que necesita planificar varios pasos | `create_deep_agent` |
| Investigador que lee/escribe archivos | `create_deep_agent` |
| Sistema multi-agente con delegación a sub-agentes | `create_deep_agent` |
| Agente tipo Claude Code o Deep Research | `create_deep_agent` |

**Regla práctica:** empieza con `create_agent`. Solo migra a `create_deep_agent` si tu tarea **realmente** necesita planning explícito, archivos o sub-agentes.